Database Connection

In [5]:
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(find_dotenv())

def connect_to_db():
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    database = os.getenv("DB_NAME")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")

    if not all([database, user, password]):
        print("Σφάλμα: Δεν βρέθηκαν οι απαραίτητες μεταβλητές στο .env!")
        return None

    try:
        connection_uri = f"postgresql://{user}:{password}@{host}:{port}/{database}"
        engine = create_engine(connection_uri)
        with engine.connect() as connection:
            print(f"Επιτυχής σύνδεση στη βάση '{database}' στο host '{host}'!")
        return engine

    except SQLAlchemyError as e:
        print(f"Σφάλμα κατά τη σύνδεση στη ΒΔ: {e}")
        return None


print("Εκκίνηση σύνδεσης με τη βάση...")
engine = connect_to_db()

if engine is None:
    print("Τερματισμός προγράμματος λόγω αποτυχίας σύνδεσης.")
    exit(1)

Εκκίνηση σύνδεσης με τη βάση...
Επιτυχής σύνδεση στη βάση 'thesis_db' στο host 'dell-micro'!


Dataframe (statsbomb_id, name, nickname)

In [3]:
import os
import json
import pandas as pd

# Το μονοπάτι (path) που βρίσκονται τα JSON αρχεία με τα lineups του StatsBomb
# Άλλαξέ το ανάλογα με το πού έχεις σώσει τα δεδομένα!
lineups_dir = '../dataset-statsbomb/data/lineups'

players_data = []
seen_player_ids = set()

print(f"Διαβάζουμε τα JSON αρχεία από: {lineups_dir} ...")

# 1. Διασχίζουμε όλα τα αρχεία .json στον φάκελο lineups
if not os.path.exists(lineups_dir):
    print(f"❌ Ο φάκελος {lineups_dir} δεν βρέθηκε. Βάλε το σωστό path!")
else:
    for filename in os.listdir(lineups_dir):
        if filename.endswith('.json'):
            filepath = os.path.join(lineups_dir, filename)

            with open(filepath, 'r', encoding='utf-8') as f:
                match_lineups = json.load(f)

                # Κάθε αρχείο lineup περιέχει συνήθως 2 ομάδες
                for team in match_lineups:

                    # Η κάθε ομάδα έχει μια λίστα 'lineup' με όλους τους παίκτες
                    for player in team.get('lineup', []):
                        player_id = player.get('player_id')

                        # Αν δεν έχουμε ξαναδεί αυτό το ID, το αποθηκεύουμε
                        if player_id not in seen_player_ids:
                            seen_player_ids.add(player_id)

                            players_data.append({
                                'uid_opendata': player_id,
                                'player_name': player.get('player_name'),
                                'player_nickname': player.get('player_nickname')
                            })

    # 2. Δημιουργία του τελικού DataFrame
    df_sb_players = pd.DataFrame(players_data)

    # Μετατρέπουμε το ID σε Int64 (για παν ενδεχόμενο, όπως κάναμε στο Understat)
    df_sb_players['uid_opendata'] = df_sb_players['uid_opendata'].astype('Int64')

    print(f"✅ Ολοκληρώθηκε! Βρέθηκαν {len(df_sb_players)} μοναδικοί παίκτες.")

    # Εμφανίζουμε τους πρώτους 5 για να δούμε πώς βγήκε
    display(df_sb_players.head(10))

Διαβάζουμε τα JSON αρχεία από: dataset-statsbomb/data/lineups ...
✅ Ολοκληρώθηκε! Βρέθηκαν 11889 μοναδικοί παίκτες.


,uid_opendata,player_name,player_nickname
0,3109,Malcom Filipe Silva de Oliveira,Malcom
1,3501,Philippe Coutinho Correia,Philippe Coutinho
2,5203,Sergio Busquets i Burgos,Sergio Busquets
3,5211,Jordi Alba Ramos,Jordi Alba
4,5213,Gerard Piqué Bernabéu,Gerard Piqué
5,5246,Luis Alberto Suárez Díaz,Luis Suárez
6,5470,Ivan Rakitić,NaN
7,5477,Ousmane Dembélé,NaN
8,5492,Samuel Yves Umtiti,Samuel Umtiti
9,5503,Lionel Andrés Messi Cuccittini,Lionel Messi


Import UID + names to DB

In [ ]:
# 1. Μετονομασία στηλών για να ταιριάζουν ακριβώς με το SQL Schema σου
df_sb_players = df_sb_players.rename(columns={
    'uid_opendata': 'uid_statsbomb',
    'player_name': 'full_name',
    'player_nickname': 'nickname'
})

# Βεβαιωνόμαστε ότι το id είναι ο σωστός τύπος για τη βάση
df_sb_players['uid_statsbomb'] = df_sb_players['uid_statsbomb'].astype('Int64')

table_name = 'statsbomb_player_info'
print(f"⏳ Ξεκινάει η εισαγωγή {len(df_sb_players)} παικτών στον πίνακα '{table_name}'...")

# 2. Εισαγωγή των δεδομένων στη βάση
try:
    with engine.begin() as conn:
        df_sb_players.to_sql(
            name=table_name,
            con=conn,
            if_exists='append',
            index=False,
            method='multi'
        )

    print(f"✅ ΕΠΙΤΥΧΙΑ! Οι παίκτες αποθηκεύτηκαν στον πίνακα '{table_name}'")

except Exception as e:
    print(f"❌ Σφάλμα κατά την εισαγωγή: {e}")

Dataframe with shots (only male + 2015/2016 season and after)

In [ ]:
# Paths (Άλλαξέ τα αν χρειάζεται)
base_dir = '../dataset-statsbomb/data'
matches_dir = os.path.join(base_dir, 'matches')
events_dir = os.path.join(base_dir, 'events')
competitions_file = os.path.join(base_dir, 'competitions.json')

# --- 1. Εύρεση των κατάλληλων διοργανώσεων (Male & >= 2015) ---
print("Διάβασμα του competitions.json για φιλτράρισμα (Male & >= 2015)...")
valid_comp_seasons = set()

if os.path.exists(competitions_file):
    with open(competitions_file, 'r', encoding='utf-8') as f:
        competitions = json.load(f)
        for comp in competitions:
            gender = comp.get('competition_gender', 'male').lower()
            season_name = comp['season_name']

            # Εξαγωγή του αρχικού έτους (π.χ. "2015/2016" -> 2015)
            start_year = int(season_name[:4])

            if gender == 'male' and start_year >= 2015:
                # Κρατάμε το ζευγάρι (competition_id, season_id)
                valid_comp_seasons.add((comp['competition_id'], comp['season_id']))
else:
    print(f"❌ Δεν βρέθηκε το αρχείο {competitions_file}.")

print(f"Βρέθηκαν {len(valid_comp_seasons)} έγκυροι συνδυασμοί διοργάνωσης-σεζόν.")


# --- 2. Εύρεση των Αγώνων (Matches) ---
target_matches = {}

for comp_id, season_id in valid_comp_seasons:
    matches_file = os.path.join(matches_dir, str(comp_id), f"{season_id}.json")

    if os.path.exists(matches_file):
        with open(matches_file, 'r', encoding='utf-8') as f:
            matches = json.load(f)
            for match in matches:
                target_matches[match['match_id']] = {
                    'competition': match['competition']['competition_name'],
                    'season': match['season']['season_name'],
                    'home_team_id': match['home_team']['home_team_id']
                }

print(f"Βρέθηκαν συνολικά {len(target_matches)} αγώνες για να ελέγξουμε.")


# --- 3. Εξαγωγή των Shots από τα Events αυτών των αγώνων ---
print("Εξαγωγή δεδομένων σουτ από τα events... (Αυτό μπορεί να πάρει λίγη ώρα)")
shots_data = []

total = len(target_matches)

for match_id, match_info in target_matches.items():

    event_file = os.path.join(events_dir, f"{match_id}.json")
    if not os.path.exists(event_file):
        continue

    with open(event_file, 'r', encoding='utf-8') as f:
        events = json.load(f)

    # Λεξικό ID -> Event για να βρίσκουμε τις ασίστ (key passes)
    events_dict = {ev['id']: ev for ev in events}

    prev_event_type = None

    for event in events:
        event_type = event['type']['name']

        if event_type == 'Shot':
            shot = event.get('shot', {})

            # Υπολογισμός h_a
            team_id = event['team']['id']
            h_a = 'h' if team_id == match_info['home_team_id'] else 'a'

            # Τοποθεσία τερματισμού (end_location)
            end_loc = shot.get('end_location', [None, None, None])
            end_x = end_loc[0] if len(end_loc) > 0 else None
            end_y = end_loc[1] if len(end_loc) > 1 else None
            end_z = end_loc[2] if len(end_loc) > 2 else None

            # Εύρεση Assister (μέσω key_pass_id)
            key_pass_id = shot.get('key_pass_id')
            player_assisted_id = None
            player_assisted_name = None

            if key_pass_id and key_pass_id in events_dict:
                pass_event = events_dict[key_pass_id]
                player_assisted_id = pass_event.get('player', {}).get('id')
                player_assisted_name = pass_event.get('player', {}).get('name')

            # Outcome
            outcome = shot.get('outcome', {}).get('name')
            is_goal = 1 if outcome == 'Goal' else 0

            shot_dict = {
                'shot_id': event['id'],
                'match_id': match_id,
                'player_id': event.get('player', {}).get('id'),

                'period': event.get('period'),
                'minute': event.get('minute'),
                'second': event.get('second'),
                'position': event.get('position', {}).get('name'),
                'team_id': team_id,
                'team_name': event.get('team', {}).get('name'),

                'competition': match_info['competition'],
                'season': match_info['season'],
                'h_a': h_a,

                'x_loc': event.get('location', [None])[0],
                'y_loc': event.get('location', [None, None])[1],

                'end_x': end_x,
                'end_y': end_y,
                'end_z': end_z,

                'play_pattern': event.get('play_pattern', {}).get('name'),
                'key_pass_id': key_pass_id,
                'last_action': prev_event_type,
                'player_assisted_id': player_assisted_id,
                'player_assisted': player_assisted_name,

                'shot_type': shot.get('type', {}).get('name'),
                'body_part': shot.get('body_part', {}).get('name'),
                'technique': shot.get('technique', {}).get('name'),
                'first_time': shot.get('first_time', False),

                'under_pressure': event.get('under_pressure', False),
                'one_on_one': shot.get('one_on_one', False),
                'open_goal': shot.get('open_goal', False),
                'aerial_won': shot.get('aerial_won', False),
                'follows_dribble': shot.get('follows_dribble', False),
                'redirect': shot.get('redirect', False),
                'deflected': shot.get('deflected', False),

                'outcome': outcome,
                'is_goal': is_goal,
                'statsbomb_xg': shot.get('statsbomb_xg', 0.0)
            }
            shots_data.append(shot_dict)

        # Αποθηκεύουμε το τρέχον event ως "προηγούμενο" για το επόμενο loop
        prev_event_type = event_type

df_sb_shots = pd.DataFrame(shots_data)
print(f"\n✅ Ολοκληρώθηκε! Εξήχθησαν συνολικά {len(df_sb_shots)} σουτ μόνο από Male διοργανώσεις (>= 2015).")
display(df_sb_shots.head())

Print DataFrame Fields

In [ ]:
print("Τα πεδία (στήλες) του df_sb_shots είναι:\n")
for i, col in enumerate(df_sb_shots.columns, 1):
    print(f"{i}. {col}")

Προετοιμασία Data Types + Column Filtering + Same order as table "statsbomb_shots"

In [ ]:
# --- ΒΗΜΑ 1: ΠΡΟΕΤΟΙΜΑΣΙΑ ΤΥΠΩΝ (DATA TYPES) ---

# 1α. Μετατροπή σε 'Int64' (για να μην γίνονται floats λόγω των NaN)
int_columns = ['match_id', 'player_id', 'period', 'minute', 'second', 'team_id', 'player_assisted_id', 'is_goal']
for col in int_columns:
    df_sb_shots[col] = df_sb_shots[col].astype('Int64')

# 1β. Μετατροπή σε float (συντεταγμένες και xG)
float_columns = ['x_loc', 'y_loc', 'end_x', 'end_y', 'end_z', 'statsbomb_xg']
for col in float_columns:
    df_sb_shots[col] = df_sb_shots[col].astype(float)

# 1γ. Μετατροπή σε Booleans (αληθές/ψευδές) για τα situational flags (γεμίζοντας τα κενά με False)
bool_columns = ['first_time', 'under_pressure', 'one_on_one', 'open_goal', 'aerial_won', 'follows_dribble', 'redirect', 'deflected']
for col in bool_columns:
    df_sb_shots[col] = df_sb_shots[col].fillna(False).astype(bool)


# --- ΒΗΜΑ 2: ΕΠΙΛΟΓΗ & ΣΕΙΡΑ ΣΤΗΛΩΝ (ΧΩΡΙΣ ΤΑ GENERATED) ---
final_columns = [
    'shot_id', 'match_id', 'player_id',
    'period', 'minute', 'second', 'position', 'team_id', 'team_name',
    'competition', 'season', 'h_a',
    'x_loc', 'y_loc',
    'end_x', 'end_y', 'end_z',
    'play_pattern', 'key_pass_id', 'last_action', 'player_assisted_id', 'player_assisted',
    'shot_type', 'body_part', 'technique', 'first_time',
    'under_pressure', 'one_on_one', 'open_goal', 'aerial_won', 'follows_dribble', 'redirect', 'deflected',
    'outcome', 'is_goal', 'statsbomb_xg'
]

# Κρατάμε ακριβώς αυτές τις στήλες, με τη συγκεκριμένη σειρά
df_sb_shots = df_sb_shots[final_columns]

# --- ΕΛΕΓΧΟΣ NOT NULL ΚΑΙ ΚΕΝΩΝ ---
print("--- ΕΛΕΓΧΟΣ ΓΙΑ NOT NULL ΠΕΔΙΑ ---")
not_null_cols = ['shot_id', 'match_id', 'player_id']
for col in not_null_cols:
    missing = df_sb_shots[col].isna().sum()
    if missing > 0:
        print(f"❌ ΠΡΟΣΟΧΗ: Η στήλη {col} έχει {missing} κενές τιμές!")
    else:
        print(f"✅ Η στήλη {col} είναι πλήρης (0 κενά).")

# Αντικατάσταση των NaN/pd.NA με None στα string πεδία (αποφεύγουμε warnings και περνάνε τέλεια ως SQL NULL)
string_cols = [
    'shot_id', 'position', 'team_name', 'competition', 'season', 'h_a',
    'play_pattern', 'key_pass_id', 'last_action', 'player_assisted',
    'shot_type', 'body_part', 'technique', 'outcome'
]

for col in string_cols:
    df_sb_shots[col] = df_sb_shots[col].where(pd.notnull(df_sb_shots[col]), None)


print("\n--- ΤΕΛΙΚΟΙ ΤΥΠΟΙ ΔΕΔΟΜΕΝΩΝ (Data Types) ---")
print(df_sb_shots.dtypes)

Εισαγωγή σούτ στη ΒΔ

In [ ]:
table_name = 'statsbomb_shots'
print(f"⏳ Ξεκινάει η εισαγωγή {len(df_sb_shots)} σουτ στον πίνακα '{table_name}'...")

try:
    # Το engine.begin() εγγυάται ότι θα γίνει commit στο τέλος
    with engine.begin() as conn:
        df_sb_shots.to_sql(
            name=table_name,
            con=conn,
            if_exists='append',
            index=False,
            chunksize=10000,
            method='multi'
        )
    print(f"✅ ΕΠΙΤΥΧΙΑ! Η εισαγωγή ολοκληρώθηκε")

except Exception as e:
    print(f"❌ Σφάλμα κατά την εισαγωγή: {e}")

Calculate xG per team per game + ppda per team per game

In [ ]:
base_dir = '../dataset-statsbomb/data'
matches_dir = os.path.join(base_dir, 'matches')
events_dir = os.path.join(base_dir, 'events')
competitions_file = os.path.join(base_dir, 'competitions.json')

print("⏳ Βήμα 1: Εύρεση Αγώνων (Male, >= 2015)...")
valid_comp_seasons = set()

if os.path.exists(competitions_file):
    with open(competitions_file, 'r', encoding='utf-8') as f:
        for comp in json.load(f):
            if comp.get('competition_gender', 'male').lower() == 'male' and int(comp['season_name'][:4]) >= 2015:
                valid_comp_seasons.add((comp['competition_id'], comp['season_id']))

target_matches = {}
for comp_id, season_id in valid_comp_seasons:
    m_file = os.path.join(matches_dir, str(comp_id), f"{season_id}.json")
    if os.path.exists(m_file):
        with open(m_file, 'r', encoding='utf-8') as f:
            for match in json.load(f):
                # Κρατάμε απλά το ID του γηπεδούχου για να ξεχωρίζουμε το home/away στα events
                target_matches[match['match_id']] = match['home_team']['home_team_id']

print(f"Βρέθηκαν {len(target_matches)} αγώνες. Ξεκινάει ο υπολογισμός xG, PPDA & Deep Completions από τα Events...")

match_data_list = []
count = 0

for match_id, home_team_id in target_matches.items():
    count += 1
    if count % 200 == 0:
        print(f"Επεξεργασία: {count} / {len(target_matches)} αγώνες...")

    event_file = os.path.join(events_dir, f"{match_id}.json")
    if not os.path.exists(event_file):
        continue

    with open(event_file, 'r', encoding='utf-8') as f:
        events = json.load(f)

    h_xg, a_xg = 0.0, 0.0
    h_deep, a_deep = 0, 0
    h_def_actions, a_def_actions = 0, 0
    h_passes_allowed, a_passes_allowed = 0, 0

    for event in events:
        team_id = event.get('team', {}).get('id')
        event_name = event.get('type', {}).get('name')
        loc = event.get('location')

        # 1. Υπολογισμός xG
        if event_name == 'Shot':
            xg = event.get('shot', {}).get('statsbomb_xg', 0.0)
            if team_id == home_team_id:
                h_xg += xg
            else:
                a_xg += xg

        if not loc:
            continue

        x = loc[0]

        # 2. Υπολογισμός PPDA (Πάσες) & Deep Completions
        if event_name == 'Pass':
            pass_info = event.get('pass', {})

            # --- PPDA ---
            # Αν η πάσα γίνεται στο επιθετικό 60% του γηπέδου (x > 48)
            if x > 48:
                if team_id == home_team_id:
                    a_passes_allowed += 1
                else:
                    h_passes_allowed += 1

            # --- DEEP COMPLETIONS ---
            # Επιτυχημένη πάσα (χωρίς outcome), όχι σέντρα (cross), με κατάληξη στα τελευταία 20 μέτρα (x >= 100)
            end_loc = pass_info.get('end_location')
            if end_loc and pass_info.get('outcome') is None and not pass_info.get('cross', False):
                if end_loc[0] >= 100:
                    if team_id == home_team_id:
                        h_deep += 1
                    else:
                        a_deep += 1

        # 3. Υπολογισμός PPDA (Αμυντικές Ενέργειες)
        is_defensive_action = False
        if event_name in ['Pressure', 'Interception', 'Block', 'Foul Committed'] or (event_name == 'Duel' and event.get('duel', {}).get('type', {}).get('name') == 'Tackle'):
            is_defensive_action = True

        if is_defensive_action and x < 72: # Άμυνα στο δικό τους 60% (x < 72)
            if team_id == home_team_id:
                h_def_actions += 1
            else:
                a_def_actions += 1

    # Υπολογισμός του τελικού κλάσματος για το PPDA (με προστασία διαίρεσης με το 0)
    h_ppda = h_passes_allowed / h_def_actions if h_def_actions > 0 else 0.0
    a_ppda = a_passes_allowed / a_def_actions if a_def_actions > 0 else 0.0

    match_data_list.append({
        'match_id': match_id,
        'home_xg': round(h_xg, 3),
        'away_xg': round(a_xg, 3),
        'home_deep': h_deep,
        'away_deep': a_deep,
        'home_ppda': round(h_ppda, 2),
        'away_ppda': round(a_ppda, 2)
    })

df_sb_stats = pd.DataFrame(match_data_list)
print(f"\n✅ Ολοκληρώθηκε! Δημιουργήθηκε το df_sb_stats με {len(df_sb_stats)} αγώνες.")
display(df_sb_stats.head())

Μatch data + xG per team + ppda per team + Deep Completions into ONE DataFrame

In [2]:
valid_comp_seasons = set()
if os.path.exists(competitions_file):
    with open(competitions_file, 'r', encoding='utf-8') as f:
        for comp in json.load(f):
            if comp.get('competition_gender', 'male').lower() == 'male' and int(comp['season_name'][:4]) >= 2015:
                valid_comp_seasons.add((comp['competition_id'], comp['season_id']))

match_data_list = []
for comp_id, season_id in valid_comp_seasons:
    m_file = os.path.join(matches_dir, str(comp_id), f"{season_id}.json")
    if os.path.exists(m_file):
        with open(m_file, 'r', encoding='utf-8') as f:
            for match in json.load(f):
                match_dt = match.get('match_date') + ' ' + match.get('match_time', '00:00:00')
                match_data_list.append({
                    'match_id': match['match_id'],
                    'match_date': match_dt,
                    'league': match['competition']['competition_name'],
                    'season': int(match['season']['season_name'][:4]),
                    'home_team': match['home_team']['home_team_name'],
                    'away_team': match['away_team']['away_team_name'],
                    'home_goals': match['home_score'],
                    'away_goals': match['away_score']
                })

df_matches_basic = pd.DataFrame(match_data_list)

# --- ΕΝΩΣΗ (MERGE) ΜΕ ΤΟ ΕΤΟΙΜΟ df_sb_stats ---
df_sb_matches = pd.merge(df_matches_basic, df_sb_stats, on='match_id', how='inner')

# Βάζουμε τις στήλες στην ακριβή σειρά του Schema σου
cols_order = [
    'match_id', 'match_date', 'league', 'season', 'home_team', 'away_team',
    'home_goals', 'away_goals', 'home_xg', 'away_xg', 'home_deep', 'away_deep',
    'home_ppda', 'away_ppda'
]
df_sb_matches = df_sb_matches[cols_order]

# Κλείδωμα των Data Types (όπως κάναμε και στο Understat) για να ανέβουν σωστά στην PostgreSQL
df_sb_matches['match_date'] = pd.to_datetime(df_sb_matches['match_date'])
int_cols = ['match_id', 'season', 'home_goals', 'away_goals', 'home_deep', 'away_deep']
for col in int_cols:
    df_sb_matches[col] = df_sb_matches[col].astype('Int64')

print(f"✅ Έτοιμο σε κλάσματα δευτερολέπτου! Το df_sb_matches ({len(df_sb_matches)} αγώνες) είναι πανέτοιμο.")
display(df_sb_matches.head())

NameError: name 'os' is not defined

Import match data to BD "statsbomb_matches"

In [18]:
table_name_sb = 'statsbomb_matches'
print(f"⏳ Δημιουργία πίνακα (αν δεν υπάρχει) και εισαγωγή {len(df_sb_matches)} αγώνων...")
try:
    with engine.begin() as conn:

        df_sb_matches.to_sql(
            name=table_name_sb,
            con=conn,
            if_exists='append',
            index=False,
            method='multi'
        )

    print(f"✅ ΕΠΙΤΥΧΙΑ! Ο πίνακας StatsBomb γέμισε.")
except Exception as e:
    print(f"❌ Σφάλμα: {e}")


⏳ Δημιουργία πίνακα (αν δεν υπάρχει) και εισαγωγή 2254 αγώνων...
✅ ΕΠΙΤΥΧΙΑ! Ο πίνακας StatsBomb γέμισε.


Import Freeze-Data from files to Dataframe

In [9]:
from sqlalchemy import text
base_dir = '../dataset-statsbomb/data'
events_dir = os.path.join(base_dir, 'events')
print("⏳ Άντληση έγκυρων shot_ids και match_ids από τη Βάση...")

# 1. Παίρνουμε όλα τα έγκυρα shot_id από τον πίνακα που έχεις ήδη φτιάξει
query_shots = "SELECT shot_id, match_id FROM statsbomb_shots"
df_valid_shots = pd.read_sql_query(query_shots, engine)

# Ομαδοποιούμε τα shot_ids ανά match_id (για αστραπιαίο indexing αργότερα)
matches_with_shots = df_valid_shots.groupby('match_id')['shot_id'].apply(set).to_dict()

print(f"Βρέθηκαν {len(df_valid_shots)} έγκυρα σουτ σε {len(matches_with_shots)} αγώνες.")
print("⏳ Εξαγωγή των Freeze Frames... (Αυτό θα πάρει 1-2 λεπτά)")

freeze_frame_data = []

for match_id, valid_shot_ids in matches_with_shots.items():

    event_file = os.path.join(events_dir, f"{match_id}.json")
    if not os.path.exists(event_file):
        continue

    with open(event_file, 'r', encoding='utf-8') as f:
        events = json.load(f)

    for event in events:
        event_id = event.get('id')

        # Αν το event είναι Shot ΚΑΙ το id του υπάρχει στα valid_shot_ids μας
        if event.get('type', {}).get('name') == 'Shot' and event_id in valid_shot_ids:

            # Παίρνουμε τη λίστα με τους παίκτες γύρω από το σουτ
            freeze_frames = event.get('shot', {}).get('freeze_frame', [])

            for player_ff in freeze_frames:
                loc = player_ff.get('location', [None, None])
                player_info = player_ff.get('player', {})
                position_info = player_ff.get('position', {})

                freeze_frame_data.append({
                    'shot_id': event_id,
                    'match_id': match_id,
                    'player_id': player_info.get('id'),
                    'position_name': position_info.get('name'),
                    'teammate': player_ff.get('teammate'),  # True = Συμπαίκτης, False = Αντίπαλος
                    'x_loc': loc[0],
                    'y_loc': loc[1]
                })

df_freeze_frames = pd.DataFrame(freeze_frame_data)
# Μετατροπή σε Int64 για να ανεβεί σωστά στην SQL
df_freeze_frames['player_id'] = df_freeze_frames['player_id'].astype('Int64')

print(f"✅ Εξήχθησαν {len(df_freeze_frames)} θέσεις παικτών από τα Freeze Frames")
display(df_freeze_frames.head(10))

⏳ Άντληση έγκυρων shot_ids και match_ids από τη Βάση...
Βρέθηκαν 56749 έγκυρα σουτ σε 2254 αγώνες.
⏳ Εξαγωγή των Freeze Frames... (Αυτό θα πάρει 1-2 λεπτά)
✅ Εξήχθησαν 735609 θέσεις παικτών από τα Freeze Frames


,shot_id,match_id,player_id,position_name,teammate,x_loc,y_loc
0,b9d7b536-849d-4d66-8c5d-12cbb4b279f7,7525,5191,Right Back,True,87.0,55.0
1,b9d7b536-849d-4d66-8c5d-12cbb4b279f7,7525,5171,Right Center Midfield,False,103.0,26.0
2,b9d7b536-849d-4d66-8c5d-12cbb4b279f7,7525,5175,Right Center Back,False,106.0,24.0
3,b9d7b536-849d-4d66-8c5d-12cbb4b279f7,7525,5170,Right Back,False,107.0,21.0
4,b9d7b536-849d-4d66-8c5d-12cbb4b279f7,7525,5193,Left Back,False,106.0,43.0
5,b9d7b536-849d-4d66-8c5d-12cbb4b279f7,7525,5172,Goalkeeper,False,118.0,41.0
6,b9d7b536-849d-4d66-8c5d-12cbb4b279f7,7525,5174,Left Center Back,False,108.0,33.0
7,b9d7b536-849d-4d66-8c5d-12cbb4b279f7,7525,5196,Center Forward,True,105.0,37.0
8,b9d7b536-849d-4d66-8c5d-12cbb4b279f7,7525,5187,Left Midfield,True,97.0,26.0
9,b9d7b536-849d-4d66-8c5d-12cbb4b279f7,7525,5177,Left Midfield,False,93.0,43.0


Εισαγωγη freeze frames στη ΒΔ

In [10]:
# --- Ανέβασμα στη Βάση ---
table_name_ff = 'statsbomb_shots_freeze_frames'

create_table_ff = f"""
DROP TABLE IF EXISTS public.{table_name_ff};
CREATE TABLE public.{table_name_ff}
(
    freeze_id SERIAL PRIMARY KEY,    -- Auto-increment ID για την κάθε θέση
    shot_id VARCHAR(36) NOT NULL,    -- Foreign Key προς statsbomb_shots
    match_id INTEGER NOT NULL,
    player_id INTEGER,
    position_name VARCHAR(100),
    teammate BOOLEAN,
    x_loc DOUBLE PRECISION,
    y_loc DOUBLE PRECISION
);
"""

print(f"⏳ Ανέβασμα του πίνακα '{table_name_ff}' στη βάση...")

try:
    with engine.begin() as conn:
        conn.execute(text(create_table_ff))

        # Το chunksize=10000 είναι απαραίτητο εδώ γιατί το DataFrame θα έχει εκατοντάδες χιλιάδες γραμμές!
        df_freeze_frames.to_sql(
            name=table_name_ff,
            con=conn,
            if_exists='append',
            index=False,
            chunksize=10000,
            method='multi'
        )
    print(f"✅ ΕΠΙΤΥΧΙΑ! Τα Freeze Frames αποθηκεύτηκαν")
except Exception as e:
    print(f"❌ Σφάλμα κατά την εισαγωγή: {e}")

⏳ Ανέβασμα του πίνακα 'statsbomb_shots_freeze_frames' στη βάση...
✅ ΕΠΙΤΥΧΙΑ! Τα Freeze Frames αποθηκεύτηκαν
